# LDaCA Text Analytics Tools on Google Colab
Run the full stack (FastAPI backend + React frontend) entirely inside Colab. Execute each cell in order. The final cell prints clickable proxied URLs to open the app.

In [ ]:
%%bash
set -euo pipefail

# System setup
sudo apt-get update -y
sudo apt-get install -y curl git

# Install Node 20 via nvm
export NVM_DIR="$HOME/.nvm"
if [ ! -d "$NVM_DIR" ]; then
  curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash
fi
source "$NVM_DIR/nvm.sh"
nvm install 20
nvm use 20
node -v
npm -v

# Clone (or update) the repo on branch 'colab'
if [ ! -d "LDaCA-Text-Analytics-Tools" ]; then
  git clone -b colab https://github.com/Australian-Text-Analytics-Platform/LDaCA-Text-Analytics-Tools.git
else
  cd LDaCA-Text-Analytics-Tools
  git fetch origin colab
  git checkout colab
  git pull
  cd ..
fi

# Initialize and update submodules (fetch each external component)
cd LDaCA-Text-Analytics-Tools
if [ -f .gitmodules ]; then
  git submodule update --init --recursive
  git submodule status || true
fi
cd ..

In [ ]:
%%bash
set -euo pipefail
cd LDaCA-Text-Analytics-Tools

# Ensure submodules are present (defensive)
if [ -f .gitmodules ]; then
  git submodule update --init --recursive
fi

# Pin IPython deps to avoid resolver warnings breaking tools
python -m pip install -U pip setuptools wheel jedi

# Verify local packages exist before installing
[ -f "docframe/pyproject.toml" ] || { echo "Missing docframe/pyproject.toml (did submodules fetch?)"; exit 1; }
[ -f "docworkspace/pyproject.toml" ] || { echo "Missing docworkspace/pyproject.toml (did submodules fetch?)"; exit 1; }
[ -f "ldaca_web_app/backend/pyproject.toml" ] || { echo "Missing backend/pyproject.toml (did submodules fetch?)"; exit 1; }

# Install the local Python packages (docframe, docworkspace, backend)
python -m pip install ./docframe
python -m pip install ./docworkspace
python -m pip install ./ldaca_web_app/backend

# Create a default .env for backend if missing
if [ -f "ldaca_web_app/backend/.env.example" ] && [ ! -f "ldaca_web_app/backend/.env" ]; then
  cp ldaca_web_app/backend/.env.example ldaca_web_app/backend/.env
fi

# Quick import sanity check
python - <<'PY'
import importlib
for m in ("docframe", "docworkspace", "ldaca_web_app_backend"):
    importlib.import_module(m)
print("Imports OK")
PY

In [ ]:
%%bash
set -euo pipefail
cd LDaCA-Text-Analytics-Tools/ldaca_web_app/frontend

# Install frontend deps (avoid postinstall scripts in Colab; re-run without the flag if you need patches)
npm ci --ignore-scripts || npm install --ignore-scripts

# Build a production bundle
CI=false PUBLIC_URL=. npm run build

# Install a simple static file server to serve the build on port 3000
npm install -g serve

In [ ]:
# Start backend (FastAPI on 8001) and serve the built frontend (3000), then print proxied URLs.
import os, subprocess, time, pathlib, sys
from google.colab import output

ROOT = pathlib.Path('LDaCA-Text-Analytics-Tools').resolve()
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

# Start backend (uvicorn)
backend = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'ldaca_web_app_backend.main:app',
    '--host', '0.0.0.0', '--port', '8001'
], cwd=str(ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Start static server for the built frontend
frontend = subprocess.Popen([
    'serve', '-s', 'ldaca_web_app/frontend/build', '-l', '3000'
], cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Allow processes to bind
time.sleep(3)

# Get Colab proxied URLs
front_url = output.eval_js('google.colab.kernel.proxyPort(3000)')
back_url  = output.eval_js('google.colab.kernel.proxyPort(8001)')
print('Frontend:', front_url)
print('Backend API docs:', back_url + 'api/docs')

# Show last few log lines from each process for quick diagnostics
def tail(proc, name, n=20):
    try:
        out = proc.stdout
        if out:
            lines = out.readlines()[-n:]
            print(f'
=== {name} (last {n} lines) ===')
            print(''.join(lines[-n:]))
    except Exception:
        pass

time.sleep(2)
tail(backend, 'Backend')
tail(frontend, 'Frontend')

## Open the app
- Click the "Frontend" link printed above to launch the UI (served via Colab’s proxy at /proxy/3000/).
- API calls are automatically routed to /proxy/8001/api by the frontend.